In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, count, when, to_json, explode
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, monotonically_increasing_id


spark.conf.set("fs.azure.account.key.xxxxx.blob.core.windows.net", "xxx")

def write_single_csv(df, final_path, tmp_path):
    """
    df → dataframe
    final_path → ruta final con nombre .csv
    tmp_path → carpeta temporal
    """

    # 1. escribir temporal
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp_path)

    # 2. localizar el part file
    files = dbutils.fs.ls(tmp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]

    # 3. mover con nombre final
    dbutils.fs.mv(part_file, final_path, True)

    # 4. borrar carpeta temporal
    dbutils.fs.rm(tmp_path, True)

In [0]:

spark = SparkSession.builder.appName("FlightsJSON").getOrCreate()

path="wasbs://bronze@xxxxx.blob.core.windows.net/flights.json"

df = spark.read.option("multiline", "true").json(path)

flights = df.select(explode("data").alias("flight")).select("flight.*")
flights = flights.withColumn("flight_id", monotonically_increasing_id())

# TABLAS
flights_main = flights.select("flight_id","flight_date","flight_status")
departure = flights.select("flight_id", col("departure.*"))
arrival = flights.select("flight_id", col("arrival.*"))
airline = flights.select("flight_id", col("airline.*"))
flight_info = flights.select(
    "flight_id",
    col("flight.number").alias("number"),
    col("flight.iata").alias("iata"),
    col("flight.icao").alias("icao")
)
aircraft = flights.select("flight_id", col("aircraft.*"))

# RUTA FINAL
base = "wasbs://silver@xxxxx.blob.core.windows.net/"
tmp  = "wasbs://silver@xxxxx.blob.core.windows.net/tmp/"

# GUARDAR CSVs
write_single_csv(flights_main, base+"flights.csv", tmp+"flights/")
write_single_csv(departure, base+"departure.csv", tmp+"departure/")
write_single_csv(arrival, base+"arrival.csv", tmp+"arrival/")
write_single_csv(airline, base+"airline.csv", tmp+"airline/")
write_single_csv(flight_info, base+"flight_info.csv", tmp+"flight_info/")
write_single_csv(aircraft, base+"aircraft.csv", tmp+"aircraft/")